# Sampling and Bootstrapping with Sets and Dictionaries

This notebook goes with today's lecture. It has three parts:

1. **What is a Jupyter notebook?** How to use this file.
2. **Sampling and bootstrapping.** The big ideas, in plain language.
3. **The functions from lecture.** The ones we were given are ready to run, with example calls you can try and change. Then you'll implement `count_means` yourself, with a description of what it should do and checks to test your work.

## Part 1: What is a Jupyter notebook?

A Jupyter notebook is a document that mixes **writing** and **code that runs**. It's made of boxes called **cells**, and each cell is one of two kinds:

- **Markdown cells** (like this one) hold text: explanations, headings and lists.
- **Code cells** hold Python. When you run one, its output appears right underneath it.

### Running a cell

Click a code cell and press **Shift + Enter**, or click the ▶ Run button. The cell runs, and the notebook moves to the next cell.

Try it on the cell below.

In [1]:
print("Hello from a code cell!")
2 + 3

Hello from a code cell!


5

You got two things back:

- The `print` output.
- `5`. A notebook automatically shows the value of the **last line** of a cell, even without `print`.

### Cells share memory

Variables and functions you make in one cell stay around for every cell you run after it. Run these two cells in order:

In [ ]:
favorite_number: int = 110

In [ ]:
print(favorite_number + 1)

### Order matters (and a common gotcha)

The notebook remembers what you **ran**, in the order you ran it, not the order the cells appear on the page. Two things follow from that:

- If you get a `NameError` saying something "is not defined", you probably skipped the cell that defines it. Scroll up and run it.
- If things get confusing, restart and run everything from the top. Look for **Kernel → Restart & Run All**, or in VS Code, **Restart** followed by **Run All**.

The number in brackets next to a code cell, like `[3]`, tells you the order cells were run in.

### Editing

Change any code cell and run it again. You can't break anything, so experiment freely! To add a cell of your own, use the **+ Code** button.

## Part 2: Sampling and bootstrapping, the big ideas

### Samples and populations

Say we want to know how many hours of sleep students in our lecture hall get. Asking all 300 people would take forever, so we ask a few.

- The **population** is everyone we want to learn about (the whole room).
- A **sample** is the small group we actually measure (say, 5 students).
- Picking the sample **at random** helps avoid bias. If we only asked the people in the front row, we might get a skewed answer.

### Two ways to draw

- **Without replacement:** like pulling names from a hat and keeping them out. No one can be picked twice, so a **set** is a natural fit.
- **With replacement:** like putting each name back before the next draw. Repeats are allowed, so we use a **list**.

### Different samples, different answers

Draw 5 random students and the average might be 7.0 hours. Draw 5 different students and it might be 6.4 or 7.4. This wobble is called **sampling variability**.

The catch is that in real life, we only get **one** sample. With just one average, how can we tell how close it is to the true average of the whole room?

### Bootstrapping

**Bootstrapping** is a clever trick for answering that question with only one sample:

1. Take your one sample, for example `[7, 5, 8, 6, 9]`.
2. Make a **bootstrap sample** by drawing 5 values from it **with replacement**. Some values repeat and some get left out, for example `[5, 9, 9, 5, 8]`.
3. Find its mean.
4. Repeat steps 2 and 3 many times (say, 1,000).

Each bootstrap sample stands in for "a new sample from the room." The **spread** of the bootstrap means shows how much our estimate could wiggle. The middle 95% of them gives a range of believable values for the real average, called a **95% bootstrap confidence interval**.

Bigger samples give narrower ranges, which means more precise estimates.

## Part 3: Implementing this in code!

Run each code cell in order, top to bottom. Most of these functions use randomness, so **your output will be different every time you run them**. That's expected!

The first few functions are written for you. Later on, you'll write `count_means` yourself, and the cells after it won't work until you do.

### `pick_names`: sampling *without* replacement (returns a set)

`random.choice(names)` returns one random item from the list. If a name gets picked again, adding it to the set changes nothing, so the loop keeps going until `picked` holds `n` different names.

In [2]:
import random


def pick_names(names: list[str], n: int) -> set[str]:
    """Pick n different names at random."""
    picked: set[str] = set()
    while len(picked) < n:
        picked.add(random.choice(names))
    return picked

In [7]:
roster: list[str] = ["Ana", "Ben", "Cy", "Dee", "Eli"]
print(pick_names(roster, 3))  # something like {'Dee', 'Ana', 'Eli'}

{'Ben', 'Eli', 'Dee'}


**Try it:** run the cell above a few times and watch the names change. What do you think happens if you ask for `pick_names(roster, 5)`? What about `pick_names(roster, 6)`? (Careful: think about that one before you run it! If it gets stuck, use the ■ Stop / Interrupt button.)

### A quick reminder: sets ignore repeats

In [ ]:
picked: set[str] = {"Ana", "Cy", "Dee", "Eli"}
picked.add("Cy")
print(len(picked))  # 4, because "Cy" was already in the set

### `resample` and `mean`: sampling *with* replacement (uses a list)

`resample` builds one bootstrap sample. It returns a **list**, because repeats are allowed. `mean` adds up the values and divides by how many there are.

In [8]:
def resample(data: list[int]) -> list[int]:
    """Draw len(data) values from data,
    putting each value back after drawing it."""
    boot: list[int] = []
    for i in range(len(data)):
        boot.append(random.choice(data))
    return boot


def mean(data: list[int]) -> float:
    total: int = 0
    for value in data:
        total += value
    return total / len(data)

In [9]:
sleep: list[int] = [7, 5, 8, 6, 9]

print(mean(sleep))             # 7.0, our one sample's average
print(mean([5, 9, 9, 5, 8]))   # 7.2

boot: list[int] = resample(sleep)
print(boot)                    # something like [5, 9, 9, 5, 8]
print(mean(boot))              # changes each run

7.0
7.2
[9, 7, 8, 9, 5]
7.6


## Your turn: implement `count_means`

Now it's your job to write the function that does the bootstrapping. With your neighbor, fill in the body of `count_means` in the cell below.

### What `count_means` should do

**Parameters**

- `data: list[int]` is our one sample, for example `[7, 5, 8, 6, 9]`.
- `trials: int` is how many bootstrap samples to make, for example `1000`.

**Returns** a `dict[float, int]`:

- Each **key** is a bootstrap mean, rounded to 1 decimal place (like `7.2`).
- Each **value** is how many times that mean came up.

**Steps.** Repeat these `trials` times:

1. Make one bootstrap sample of `data` using `resample`.
2. Find its mean using `mean`, then round it with `round(m, 1)` so the keys stay tidy.
3. Add 1 to that mean's count in your dictionary. If the mean is already a key, add 1 to its count; otherwise, create the key with a count of 1.

When the loop is done, return the dictionary.

**Things that should be true when you're done**

- The values in the returned dictionary add up to exactly `trials`, since every trial adds 1 to some count.
- Every key is between the smallest and largest value in `data`. For our sample, that's 5 to 9.
- `count_means(sleep, 1000)` gives something like `{7.2: 120, 6.4: 77, 8.0: 37, ...}`. The exact numbers change every run.

**Don't change** the function name, its parameters or its return type. The tests below rely on them.

In [ ]:
def count_means(data: list[int], trials: int) -> dict[float, int]:
    """Make trials bootstrap samples of data. Return a dict whose
    keys are bootstrap means (rounded to 1 decimal place) and whose
    values are how many times each mean came up."""
    # TODO: replace the line below with your implementation
    ...

<details>
<summary><b>Stuck? Click here for a hint.</b></summary>

Fill in the four blanks:

```python
def count_means(data: list[int], trials: int) -> dict[float, int]:
    counts: dict[float, int] = {}
    for t in range(____):
        boot: list[int] = resample(____)
        m: float = round(mean(boot), 1)
        if ____ in counts:
            counts[m] += 1
        else:
            ____
    return counts
```

The loop runs once per trial. Each trial makes one bootstrap sample, finds its mean, and adds 1 to that mean's count.
</details>

### Check your work

Run the cell below once you've written `count_means`. It prints a ✅ for each check that passes. If you see an error, read the message, fix your function, **rerun the `count_means` cell**, then rerun this one.

In [ ]:
result = count_means(sleep, 1000)

assert isinstance(result, dict), "count_means should return a dict. Did you forget to return counts?"
print("✅ returns a dictionary")

total: int = 0
for key in result:
    total += result[key]
assert total == 1000, f"the counts should add up to 1000 (the number of trials), but they add up to {total}"
print("✅ counts add up to the number of trials")

for key in result:
    assert 5 <= key <= 9, f"{key} can't be a mean of values from {sleep}"
    assert key == round(key, 1), f"{key} isn't rounded to 1 decimal place"
print("✅ every key is a rounded mean in the right range")

assert count_means(sleep, 0) == {}, "with 0 trials, the dictionary should be empty"
print("✅ 0 trials gives an empty dictionary")

print("All checks passed. Nice work!")

### Try out your function

In [ ]:
counts: dict[float, int] = count_means(sleep, 1000)
print(counts)
print(counts[7.0])  # around 120 (it changes each run)

## Drawing the histogram

A dictionary of counts is hard to read at a glance, so let's draw it as a bar chart, also called a **histogram**. There's one bar for each bootstrap mean, and its height is how many times that mean came up. This is the same picture as the slide "1,000 bootstrap means from our one sample."

The bars in the **middle 95%** are colored lavender, and the few percent left over at each end are gray. The lavender range is our **95% bootstrap confidence interval**: a range of believable values for the whole room's average.

These functions are written for you. They need your `count_means` to work.

> The plot uses a library called **matplotlib**. If you get `ModuleNotFoundError: No module named 'matplotlib'`, run `%pip install matplotlib` in a new code cell, then restart the kernel.

In [ ]:
import matplotlib.pyplot as plt


def middle_95(counts: dict[float, int]) -> tuple[float, float]:
    """Return the (low, high) range of keys that holds the middle 95%
    of all the counts, leaving about 2.5% out at each end."""
    total: int = 0
    for key in counts:
        total += counts[key]
    low: float = 0.0
    high: float = 0.0
    found_low: bool = False
    so_far: int = 0
    for key in sorted(counts):
        so_far += counts[key]
        if not found_low and so_far > 0.025 * total:
            low = key
            found_low = True
        if so_far >= 0.975 * total:
            high = key
            return (low, high)
    return (low, high)


def plot_means(counts: dict[float, int]) -> None:
    """Draw counts as a histogram, coloring the middle 95% lavender."""
    low, high = middle_95(counts)
    keys: list[float] = sorted(counts)
    heights: list[int] = []
    colors: list[str] = []
    for key in keys:
        heights.append(counts[key])
        if low <= key <= high:
            colors.append("#C9BFF2")  # lavender: the middle 95%
        else:
            colors.append("#4A4A4A")  # gray: the ends

    # Make each bar a little narrower than the gap between neighboring means.
    width: float = 0.18
    for i in range(1, len(keys)):
        gap: float = keys[i] - keys[i - 1]
        if gap * 0.9 < width:
            width = gap * 0.9

    total: int = 0
    for key in counts:
        total += counts[key]

    fig, ax = plt.subplots(figsize=(9, 4.5), facecolor="#212121")
    ax.set_facecolor("#212121")
    ax.bar(keys, heights, width=width, color=colors)
    ax.set_title(f"{total:,} bootstrap means from our one sample", color="white", fontsize=15, loc="left")
    ax.set_xlabel("bootstrap mean", color="white")
    ax.set_ylabel("how many times", color="white")
    ax.tick_params(colors="white")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#888888")
    plt.show()
    print(f"Middle 95% of bootstrap means: {low} to {high}")

### Bootstrap 1,000 times and plot

Run this cell a few times. The bars shift a little each run, but the lavender range should stay close to **5.8 to 8.2 hours**, the range from lecture.

In [ ]:
sleep: list[int] = [7, 5, 8, 6, 9]
counts: dict[float, int] = count_means(sleep, 1000)
plot_means(counts)

### What if we had asked 20 students?

`sleep * 4` makes a list of 20 values with the same mix of answers as our 5. With more data, the bootstrap means bunch up more tightly, so the range should come out narrower, around **6.4 to 7.6**.

Don't worry if the bars zigzag up and down. Means of 20 values land on spots like 6.85, and `round(m, 1)` has to push each one to 6.8 or 6.9. Computers store decimals like 6.85 a tiny bit high or low, so some bars collect extra means and their neighbors collect fewer. Focus on the overall shape and the lavender range.

In [ ]:
sleep_20: list[int] = sleep * 4
counts_20: dict[float, int] = count_means(sleep_20, 1000)
plot_means(counts_20)

### Bonus: which values made it into a bootstrap sample?

Turning a bootstrap sample into a **set** removes the repeats, so it shows which of the original values were picked at least once.

In [ ]:
boot: list[int] = resample(sleep)
print(boot)            # something like [8, 8, 5, 9, 8]
print(set(boot))       # something like {8, 9, 5}
print(len(set(boot)))  # how many different values made it in

On average, a bootstrap sample of our 5 values contains only about 3.4 of them. For big samples, it's about 63% of the original values.

## Things to try

- Change `sleep` to your own list of numbers and rerun the cells below it.
- Try `count_means(sleep, 10)` versus `count_means(sleep, 10000)`. How does the dictionary change?
- Try `plot_means(count_means(sleep, 50))`. With only 50 trials, how does the histogram look?
- Try a sample of 100 students with `sleep * 20`. How narrow does the range get?